# MOSAIC Pretrained Prefill Correctness Rerun

This notebook reruns shared-prefix prefill measurements after correcting cache-format handling and explicit position/mask inputs. It evaluates GPT-2 Small, SmolLM2-135M, and TinyLlama-1.1B and records latency, logit error, cosine similarity, top-1 agreement, and KL divergence.

In [ ]:
import os
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "transformers>=4.57,<5",
    "accelerate>=1.10,<2",
    "safetensors>=0.6,<1",
    "sentencepiece>=0.2,<1",
    "psutil>=5.9,<8",
])

In [ ]:
import copy
import gc
import inspect
import json
import math
import platform
import time
import warnings
import zipfile
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache

warnings.filterwarnings("ignore")
torch.set_float32_matmul_precision("high")

ROOT = Path("/kaggle/working/mosaic_prefill_rerun")
RESULTS = ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("Kaggle GPU accelerator must be enabled.")

DTYPE = torch.float16
MODELS = {
    "GPT-2 Small": "openai-community/gpt2",
    "SmolLM2-135M": "HuggingFaceTB/SmolLM2-135M",
    "TinyLlama-1.1B": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
}
CONTEXTS = [256, 512, 1024, 2048]
REPEATS = 20

print("GPU:", torch.cuda.get_device_name(0))
print("transformers and torch:", torch.__version__)

In [ ]:
SYSTEM_PROMPT = (
    "You are a repository analysis agent. Preserve exact symbol names, cite file paths, "
    "and never invent test results. Use tools only when necessary."
)

TOOL_SCHEMA = json.dumps({
    "tools": [
        {
            "name": "search_repository",
            "description": "Search source files by query.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string"},
                    "path": {"type": "string"},
                    "limit": {"type": "integer"},
                },
                "required": ["query"],
            },
        },
        {
            "name": "run_tests",
            "description": "Run a test target.",
            "parameters": {
                "type": "object",
                "properties": {
                    "target": {"type": "string"},
                    "timeout": {"type": "integer"},
                },
                "required": ["target"],
            },
        },
    ]
}, indent=2)

AGENT_MEMORY = (
    "Persistent project memory: Python 3.11; preserve public APIs; "
    "migrations require rollback; tests use pytest and mypy; "
    "performance regressions above five percent block release."
)

def collect_repository_text(limit=100_000):
    roots = [Path(torch.__file__).resolve().parent, Path(pd.__file__).resolve().parent]
    parts = []
    total = 0
    for root in roots:
        for file in root.rglob("*.py"):
            try:
                text = file.read_text(encoding="utf-8", errors="ignore")
            except Exception:
                continue
            piece = f"\n# FILE: {file.relative_to(root)}\n{text}\n"
            parts.append(piece)
            total += len(piece)
            if total >= limit:
                return "".join(parts)[:limit]
    return "".join(parts)[:limit]

REPOSITORY = collect_repository_text()
RAG = "\n\n".join([
    "A cache stores reusable computation keyed by an exact content identity.",
    "A compiler can lower repeated sequence segments into closed state-transition operators.",
    "Prefix KV caching reuses only a shared causal prefix and does not generally reuse arbitrary internal substrings.",
    "Floating-point reassociation can change low-order bits even when two executions are algebraically equivalent.",
] * 64)

QUERIES = [
    "Find the shape validation path.",
    "Explain retry behavior.",
    "Identify a possible memory leak.",
    "List filesystem functions.",
    "Propose error-handling tests.",
    "Explain configuration precedence.",
    "Locate serialization.",
    "Explain caching.",
]

WORKLOADS = []
for i, query in enumerate(QUERIES):
    WORKLOADS.extend([
        {"workload": "system_prompt", "request_id": f"s{i}", "text": SYSTEM_PROMPT + "\nUser: " + query},
        {"workload": "tool_schema", "request_id": f"t{i}", "text": SYSTEM_PROMPT + "\n" + TOOL_SCHEMA + "\nUser: " + query},
        {"workload": "repository", "request_id": f"r{i}", "text": SYSTEM_PROMPT + "\n" + REPOSITORY + "\nUser: " + query},
        {"workload": "agent_memory", "request_id": f"m{i}", "text": SYSTEM_PROMPT + "\n" + AGENT_MEMORY + "\nUser: " + query},
        {"workload": "rag", "request_id": f"g{i}", "text": SYSTEM_PROMPT + "\n" + RAG + "\nUser: " + query},
    ])

In [ ]:
def sync():
    torch.cuda.synchronize()

def parameter_count(model):
    return sum(p.numel() for p in model.parameters())

def maximum_context(model):
    for name in ("max_position_embeddings", "n_positions", "max_sequence_length", "seq_length"):
        value = getattr(model.config, name, None)
        if value:
            return int(value)
    return 2048

def longest_common_prefix(sequences):
    if not sequences:
        return 0
    limit = min(map(len, sequences))
    for i in range(limit):
        if any(sequence[i] != sequences[0][i] for sequence in sequences[1:]):
            return i
    return limit

def analyze_prefix_reuse(tokenizer, label):
    groups = defaultdict(list)
    for item in WORKLOADS:
        groups[item["workload"]].append(
            tokenizer.encode(item["text"], add_special_tokens=False)
        )

    rows = []
    for workload, sequences in groups.items():
        prefix = longest_common_prefix(sequences)
        total = sum(map(len, sequences))
        reusable = prefix * (len(sequences) - 1)
        rows.append({
            "tokenizer": label,
            "workload": workload,
            "requests": len(sequences),
            "shared_prefix_tokens": prefix,
            "total_request_tokens": total,
            "cross_request_reusable_fraction": reusable / max(1, total),
        })
    return rows

def legacy_view(cache):
    if hasattr(cache, "to_legacy_cache"):
        return cache.to_legacy_cache()
    return cache

def fresh_cache(model, frozen_legacy):
    model_type = getattr(model.config, "model_type", "")
    if model_type == "gpt2":
        return frozen_legacy
    return DynamicCache.from_legacy_cache(frozen_legacy)

def filtered_kwargs(model, kwargs):
    signature = inspect.signature(model.forward)
    return {key: value for key, value in kwargs.items() if key in signature.parameters}

def timed_call(factory, warmups=3, repeats=20):
    for _ in range(warmups):
        factory()
    sync()
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        factory()
        sync()
        times.append((time.perf_counter() - start) * 1000)
    return {
        "median_ms": float(np.median(times)),
        "mean_ms": float(np.mean(times)),
        "sd_ms": float(np.std(times, ddof=1)),
    }

def logit_metrics(full_logits, cached_logits):
    full = full_logits.float()
    cached = cached_logits.float()
    delta = full - cached
    p = F.log_softmax(full, dim=-1)
    q = F.log_softmax(cached, dim=-1)
    kl = F.kl_div(q, p.exp(), reduction="batchmean", log_target=False)
    return {
        "max_abs_logit_error": float(delta.abs().max()),
        "mean_abs_logit_error": float(delta.abs().mean()),
        "cosine_similarity": float(F.cosine_similarity(full, cached, dim=-1).mean()),
        "top1_agreement": float((full.argmax(-1) == cached.argmax(-1)).float().mean()),
        "kl_full_to_cached": float(kl),
    }

In [ ]:
@torch.no_grad()
def benchmark_model(label, model_id):
    print("\nLoading", label, model_id)
    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    load_kwargs = {
        "torch_dtype": DTYPE,
        "low_cpu_mem_usage": True,
    }
    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            attn_implementation="sdpa",
            **load_kwargs,
        )
    except Exception:
        model = AutoModelForCausalLM.from_pretrained(model_id, **load_kwargs)

    model.to(DEVICE)
    model.eval()

    prefix_rows = analyze_prefix_reuse(tokenizer, label)
    max_ctx = maximum_context(model)
    base_text = SYSTEM_PROMPT + "\n" + TOOL_SCHEMA + "\n" + AGENT_MEMORY + "\n" + REPOSITORY
    base_ids = tokenizer(base_text, return_tensors="pt", add_special_tokens=False).input_ids[0]
    suffix_ids_all = tokenizer(
        "\nUser: Explain the caching behavior.\nAssistant:",
        return_tensors="pt",
        add_special_tokens=False,
    ).input_ids[0]

    rows = []
    seen_actual_lengths = set()

    for requested_context in CONTEXTS:
        suffix_ids = suffix_ids_all[:16]
        target = min(requested_context, max_ctx)
        prefix_length = max(32, target - suffix_ids.numel())
        prefix_ids = base_ids[:prefix_length]
        actual_length = int(prefix_ids.numel() + suffix_ids.numel())

        if actual_length in seen_actual_lengths:
            continue
        seen_actual_lengths.add(actual_length)

        prefix_batch = prefix_ids.unsqueeze(0).to(DEVICE)
        suffix_batch = suffix_ids.unsqueeze(0).to(DEVICE)
        full_batch = torch.cat([prefix_batch, suffix_batch], dim=1)

        full_attention = torch.ones_like(full_batch)
        prefix_attention = torch.ones_like(prefix_batch)
        suffix_attention = torch.ones(
            (1, actual_length),
            dtype=torch.long,
            device=DEVICE,
        )
        suffix_positions = torch.arange(
            prefix_batch.shape[1],
            actual_length,
            device=DEVICE,
        ).unsqueeze(0)
        cache_position = suffix_positions[0]

        full_kwargs = filtered_kwargs(model, {
            "input_ids": full_batch,
            "attention_mask": full_attention,
            "use_cache": False,
        })
        prefix_kwargs = filtered_kwargs(model, {
            "input_ids": prefix_batch,
            "attention_mask": prefix_attention,
            "use_cache": True,
        })

        prefix_output = model(**prefix_kwargs)
        frozen_legacy = legacy_view(prefix_output.past_key_values)

        def full_call():
            return model(**full_kwargs).logits[:, -1, :]

        def cached_call():
            cache = fresh_cache(model, frozen_legacy)
            kwargs = filtered_kwargs(model, {
                "input_ids": suffix_batch,
                "attention_mask": suffix_attention,
                "position_ids": suffix_positions,
                "cache_position": cache_position,
                "past_key_values": cache,
                "use_cache": True,
            })
            return model(**kwargs).logits[:, -1, :]

        full_logits = full_call()
        cached_logits = cached_call()
        metrics = logit_metrics(full_logits, cached_logits)

        full_time = timed_call(full_call, repeats=REPEATS)
        cached_time = timed_call(cached_call, repeats=REPEATS)

        row = {
            "model": label,
            "model_id": model_id,
            "parameters": parameter_count(model),
            "requested_context": requested_context,
            "actual_context_tokens": actual_length,
            "prefix_tokens": int(prefix_batch.shape[1]),
            "suffix_tokens": int(suffix_batch.shape[1]),
            "full_prefill_median_ms": full_time["median_ms"],
            "cached_suffix_median_ms": cached_time["median_ms"],
            "warm_speedup": full_time["median_ms"] / cached_time["median_ms"],
            **metrics,
        }
        print(row)
        rows.append(row)

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return rows, prefix_rows

In [ ]:
benchmark_rows = []
prefix_rows = []
failures = []

for label, model_id in MODELS.items():
    try:
        model_rows, model_prefix_rows = benchmark_model(label, model_id)
        benchmark_rows.extend(model_rows)
        prefix_rows.extend(model_prefix_rows)
    except Exception as error:
        failures.append({
            "model": label,
            "model_id": model_id,
            "error_type": type(error).__name__,
            "error": repr(error),
        })
        print("FAILED:", label, repr(error))
        gc.collect()
        torch.cuda.empty_cache()

benchmark_df = pd.DataFrame(benchmark_rows)
prefix_df = pd.DataFrame(prefix_rows)
failures_df = pd.DataFrame(failures)

benchmark_df.to_csv(RESULTS / "pretrained_prefill_corrected.csv", index=False)
prefix_df.to_csv(RESULTS / "workload_prefix_reuse_corrected.csv", index=False)
failures_df.to_csv(RESULTS / "failures.csv", index=False)

display(benchmark_df)
display(prefix_df)
display(failures_df)

## Correctness checks

Top-1 agreement must equal 1.0. Cosine similarity and KL divergence are reported as numerical diagnostics. Nonzero FP16 or SDPA logit differences are not treated as semantic failures by themselves. Failed models are excluded from reported results.

In [ ]:
if not benchmark_df.empty:
    verdict = benchmark_df.assign(
        correctness_pass=(
            (benchmark_df["top1_agreement"] == 1.0)
            & (benchmark_df["cosine_similarity"] > 0.999)
            & (benchmark_df["kl_full_to_cached"] < 1e-3)
        )
    )
    verdict.to_csv(RESULTS / "correctness_verdict.csv", index=False)
    display(verdict)

    for model_name, group in benchmark_df.groupby("model"):
        import matplotlib.pyplot as plt
        plt.plot(
            group["actual_context_tokens"],
            group["warm_speedup"],
            marker="o",
            label=model_name,
        )
    plt.xlabel("Context tokens")
    plt.ylabel("Full prefill / cached suffix")
    plt.title("Corrected shared-prefix KV-cache benchmark")
    plt.legend()
    plt.tight_layout()
    plt.savefig(RESULTS / "prefill_speedup_corrected.png", dpi=180)
    plt.show()

In [ ]:
manifest = {
    "completed_utc": pd.Timestamp.utcnow().isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "transformers": __import__("transformers").__version__,
    "gpu": torch.cuda.get_device_name(0),
    "gpu_memory": torch.cuda.get_device_properties(0).total_memory,
    "dtype": str(DTYPE),
    "contexts": CONTEXTS,
    "repeats": REPEATS,
}
(RESULTS / "environment.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

zip_path = Path("/kaggle/working/mosaic_prefill_rerun_results.zip")
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for file in ROOT.rglob("*"):
        if file.is_file():
            archive.write(file, file.relative_to(ROOT))

print("Download:", zip_path)

## Outputs

Save the executed notebook and `/kaggle/working/mosaic_prefill_rerun_results.zip`.